In [ ]:
from google.colab import files
import pandas as pd
import io

uploaded_train = files.upload()

student_dataset = pd.read_csv(
    io.BytesIO(uploaded_train[list(uploaded_train.keys())[0]])
)

display(student_dataset.head())
print("Shape:", student_dataset.shape)


Saving student_dataset.csv to student_dataset.csv


,unit_id,unit_type,ambient_temp,process_temp,coolant_flow,shaft_speed,drive_torque,power_load,component_wear,thermal_stress,mechanical_load,vibration_index,condition
0,UNT-184174,M,21.5,56.4,17.85,1454.0,58.2,9.099,30.0,29.31,57.19,1.768,0
1,UNT-236234,H,29.9,65.0,30.88,1720.0,54.6,10.101,13.0,23.93,36.94,1.081,0
2,UNT-247460,M,26.4,47.8,19.75,1552.0,38.4,6.328,89.0,13.52,38.22,1.031,0
3,UNT-176316,M,28.8,60.9,19.10,1513.0,49.1,7.768,59.0,24.62,46.17,1.011,0
4,UNT-105110,S,29.4,64.0,11.77,1536.0,52.6,8.405,36.0,27.83,94.72,1.851,1


Shape: (16000, 13)


In [ ]:

uploaded_test = files.upload()

test = pd.read_csv(
    io.BytesIO(uploaded_test[list(uploaded_test.keys())[0]])
)

display(test.head())
print("Shape:", test.shape)


Saving test.csv to test.csv


,unit_id,unit_type,ambient_temp,process_temp,coolant_flow,shaft_speed,drive_torque,power_load,component_wear,thermal_stress,mechanical_load,vibration_index
0,UNT-184630,S,23.7,51.2,13.22,1345.0,48.5,6.692,155.0,22.60,82.43,1.750
1,UNT-228473,S,31.8,58.9,14.32,1459.0,48.7,7.390,169.0,20.53,94.53,1.740
2,UNT-255783,M,28.0,83.4,19.20,1626.0,77.5,14.326,94.0,49.25,71.09,1.840
3,UNT-119059,S,31.4,65.7,14.35,1513.0,62.4,9.542,165.0,32.32,137.84,2.563
4,UNT-205559,S,31.9,59.2,13.19,1198.0,49.5,6.109,78.0,18.45,94.00,1.697


Shape: (4000, 12)


In [ ]:

print("Training data information:")
student_dataset.info()

print("\nMissing values:")
print(student_dataset.isnull().sum())

print("\nDuplicate rows:", student_dataset.duplicated().sum())

print("\nTarget distribution:")
print(student_dataset["condition"].value_counts())

print("\nTarget proportions:")
print(student_dataset["condition"].value_counts(normalize=True).round(4))

print("\nUnique unit types:")
print(student_dataset["unit_type"].value_counts())


Training data information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   unit_id          16000 non-null  object 
 1   unit_type        15978 non-null  object 
 2   ambient_temp     15953 non-null  float64
 3   process_temp     15955 non-null  float64
 4   coolant_flow     15939 non-null  float64
 5   shaft_speed      15968 non-null  float64
 6   drive_torque     15964 non-null  float64
 7   power_load       15975 non-null  float64
 8   component_wear   15944 non-null  float64
 9   thermal_stress   15950 non-null  float64
 10  mechanical_load  15955 non-null  float64
 11  vibration_index  15926 non-null  float64
 12  condition        16000 non-null  int64  
dtypes: float64(10), int64(1), object(2)
memory usage: 1.6+ MB

Missing values:
unit_id             0
unit_type          22
ambient_temp       47
process_temp       45
coolan

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Separate features and target
X = student_dataset.drop(columns=["condition", "unit_id"])
y = student_dataset["condition"]

X_test = test.drop(columns=["unit_id"])

# Identify numerical and categorical columns
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(include="object").columns.tolist()

# Preprocessing pipelines
preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

# Split the data for validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training split:", X_train.shape)
print("Validation split:", X_val.shape)
print("Test dataset:", X_test.shape)

print("\nTraining class proportions:")
print(y_train.value_counts(normalize=True).sort_index().round(3))

print("\nNumerical features:", num_cols)
print("Categorical features:", cat_cols)


Training split: (12800, 11)
Validation split: (3200, 11)
Test dataset: (4000, 11)

Training class proportions:
condition
0    0.50
1    0.25
2    0.25
Name: proportion, dtype: float64

Numerical features: ['ambient_temp', 'process_temp', 'coolant_flow', 'shaft_speed', 'drive_torque', 'power_load', 'component_wear', 'thermal_stress', 'mechanical_load', 'vibration_index']
Categorical features: ['unit_type']


XG BOOST (baseline)

In [ ]:

%pip -q install xgboost


In [ ]:

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.pipeline import Pipeline

xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    ))
])

# Train the model
xgb_model.fit(X_train, y_train)

# Validate
val_pred = xgb_model.predict(X_val)

print("Validation Accuracy:",
      round(accuracy_score(y_val, val_pred), 4))

print("Validation Macro F1:",
      round(f1_score(y_val, val_pred, average="macro"), 4))

print("\nClassification Report:")
print(classification_report(y_val, val_pred))


Validation Accuracy: 0.8934
Validation Macro F1: 0.8832

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.94      0.92      1600
           1       0.88      0.83      0.86       800
           2       0.90      0.85      0.87       800

    accuracy                           0.89      3200
   macro avg       0.89      0.88      0.88      3200
weighted avg       0.89      0.89      0.89      3200



Feature engineering

In [ ]:

import numpy as np
import joblib
import os

# Folder for saving all models and submissions
os.makedirs("saved_models", exist_ok=True)
os.makedirs("submissions", exist_ok=True)

# Feature engineering function
def add_features(df):
    df = df.copy()

    df["temperature_difference"] = (
        df["process_temp"] - df["ambient_temp"]
    )

    df["speed_torque_product"] = (
        df["shaft_speed"] * df["drive_torque"]
    )

    df["torque_speed_ratio"] = (
        df["drive_torque"] / df["shaft_speed"].replace(0, np.nan)
    )

    df["wear_speed_ratio"] = (
        df["component_wear"] /
        df["shaft_speed"].replace(0, np.nan)
    )

    return df

# Apply to training features and test features
X_fe = add_features(X)
X_test_fe = add_features(X_test)

# Preserve the same validation split
X_train_fe = X_fe.loc[X_train.index]
X_val_fe = X_fe.loc[X_val.index]

print("Original feature count:", X.shape[1])
print("Feature count after engineering:", X_fe.shape[1])
print("\nNew features:")
print([
    "temperature_difference",
    "speed_torque_product",
    "torque_speed_ratio",
    "wear_speed_ratio"
])

print("\nTraining features shape:", X_train_fe.shape)
print("Validation features shape:", X_val_fe.shape)
print("Test features shape:", X_test_fe.shape)


Original feature count: 11
Feature count after engineering: 15

New features:
['temperature_difference', 'speed_torque_product', 'torque_speed_ratio', 'wear_speed_ratio']

Training features shape: (12800, 15)
Validation features shape: (3200, 15)
Test features shape: (4000, 15)


Train, evaluate, and save XGBoost with engineered features

In [ ]:

from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier
import joblib

# Build a fresh preprocessing pipeline
preprocessor_fe = clone(preprocessor)

xgb_fe = Pipeline([
    ("preprocessor", preprocessor_fe),
    ("model", XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    ))
])

# Train
xgb_fe.fit(X_train_fe, y_train)

# Validate
val_pred_fe = xgb_fe.predict(X_val_fe)

accuracy_fe = accuracy_score(y_val, val_pred_fe)
macro_f1_fe = f1_score(y_val, val_pred_fe, average="macro")

print("Feature-engineered XGBoost")
print("Validation Accuracy:", round(accuracy_fe, 4))
print("Validation Macro F1:", round(macro_f1_fe, 4))

# Save the fitted model and preprocessing together
joblib.dump(xgb_fe, "saved_models/xgboost_feature_engineered.joblib")

# Predict the unlabeled test dataset
test_pred_fe = xgb_fe.predict(X_test_fe)

# Create submission (we will verify the required format later)
submission_fe = pd.DataFrame({
    "condition": test_pred_fe
})

submission_fe.to_csv(
    "submissions/xgboost_feature_engineered.csv",
    index=False
)

print("\nModel saved successfully.")
print("Submission saved successfully.")
print("Submission shape:", submission_fe.shape)
print("\nPredicted class counts:")
print(submission_fe["condition"].value_counts().sort_index())


Feature-engineered XGBoost
Validation Accuracy: 0.8934
Validation Macro F1: 0.8832

Model saved successfully.
Submission saved successfully.
Submission shape: (4000, 1)

Predicted class counts:
condition
0    2119
1     944
2     937
Name: count, dtype: int64


In [ ]:
import os
import joblib
import pandas as pd

os.makedirs("saved_models", exist_ok=True)
os.makedirs("submissions", exist_ok=True)

# Save the baseline model
joblib.dump(xgb_model, "saved_models/xgboost_baseline.joblib")

# Predict on the original test features
test_pred_baseline = xgb_model.predict(X_test)

# Correct Kaggle submission format
submission_baseline = pd.DataFrame({
    "unit_id": test["unit_id"],
    "condition": test_pred_baseline
})

submission_baseline.to_csv(
    "submissions/xgboost_baseline.csv",
    index=False
)

print("Baseline model saved.")
print("\nSubmission preview:")
display(submission_baseline.head())

print("Submission shape:", submission_baseline.shape)
print("Required columns:", submission_baseline.columns.tolist())
print("Missing predictions:", submission_baseline["condition"].isna().sum())


Baseline model saved.

Submission preview:


,unit_id,condition
0,UNT-184630,0
1,UNT-228473,0
2,UNT-255783,1
3,UNT-119059,2
4,UNT-205559,0


Submission shape: (4000, 2)
Required columns: ['unit_id', 'condition']
Missing predictions: 0


In [ ]:
import os
import joblib

os.makedirs("saved_models", exist_ok=True)

joblib.dump(
    xgb_model,
    "saved_models/xgboost_baseline.joblib"
)

joblib.dump(
    xgb_fe,
    "saved_models/xgboost_feature_engineered.joblib"
)

print("Saved models:")
print("- xgboost_baseline.joblib")
print("- xgboost_feature_engineered.joblib")


Saved models:
- xgboost_baseline.joblib
- xgboost_feature_engineered.joblib


CAT BOOST

In [ ]:
%pip -q install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.2 MB/s eta 0:00:00


In [ ]:

from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score
import joblib

cat_features = ["unit_type"]

catboost_model = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    loss_function="MultiClass",
    eval_metric="Accuracy",
    random_seed=42,
    verbose=200,
    allow_writing_files=False
)

# Prepare categorical values and missing values
X_train_cat = X_train.copy()
X_val_cat = X_val.copy()

for col in cat_features:
    X_train_cat[col] = X_train_cat[col].fillna("Unknown").astype(str)
    X_val_cat[col] = X_val_cat[col].fillna("Unknown").astype(str)

# CatBoost handles missing numerical values natively
catboost_model.fit(
    X_train_cat,
    y_train,
    cat_features=cat_features,
    eval_set=(X_val_cat, y_val),
    early_stopping_rounds=100,
    verbose=200
)

# Evaluate on validation data
val_pred_cat = catboost_model.predict(X_val_cat).ravel()

cat_accuracy = accuracy_score(y_val, val_pred_cat)
cat_f1 = f1_score(y_val, val_pred_cat, average="macro")

print("\nCatBoost Validation Accuracy:", round(cat_accuracy, 4))
print("CatBoost Validation Macro F1:", round(cat_f1, 4))

# Save the trained model
catboost_model.save_model("saved_models/catboost_baseline.cbm")

print("\nCatBoost model saved successfully.")


0:	learn: 0.7659375	test: 0.7746875	best: 0.7746875 (0)	total: 143ms	remaining: 2m 23s
200:	learn: 0.8869531	test: 0.8850000	best: 0.8850000 (197)	total: 11.6s	remaining: 46s
400:	learn: 0.9017188	test: 0.8900000	best: 0.8912500 (308)	total: 17.8s	remaining: 26.5s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.89125
bestIteration = 308

Shrink model to first 309 iterations.

CatBoost Validation Accuracy: 0.8912
CatBoost Validation Macro F1: 0.881

CatBoost model saved successfully.


LIGHT GBM

In [ ]:
%pip -q install lightgbm

In [ ]:

from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score
import joblib

lightgbm_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", LGBMClassifier(
        objective="multiclass",
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ))
])

# Train
lightgbm_model.fit(X_train, y_train)

# Validate
val_pred_lgbm = lightgbm_model.predict(X_val)

lgbm_accuracy = accuracy_score(y_val, val_pred_lgbm)
lgbm_f1 = f1_score(y_val, val_pred_lgbm, average="macro")

print("LightGBM Validation Accuracy:", round(lgbm_accuracy, 4))
print("LightGBM Validation Macro F1:", round(lgbm_f1, 4))

# Save the entire pipeline
joblib.dump(
    lightgbm_model,
    "saved_models/lightgbm_baseline.joblib"
)

print("\nLightGBM model saved successfully.")


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM Validation Accuracy: 0.8878
LightGBM Validation Macro F1: 0.8772

LightGBM model saved successfully.


In [ ]:

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

# Collect all validation predictions
model_predictions = {
    "XGBoost Baseline": val_pred,
    "XGBoost Feature Engineered": val_pred_fe,
    "CatBoost": val_pred_cat,
    "LightGBM": val_pred_lgbm
}

# Collect fitted models for training accuracy
fitted_models = {
    "XGBoost Baseline": (xgb_model, X_train, y_train),
    "XGBoost Feature Engineered": (xgb_fe, X_train_fe, y_train),
    "CatBoost": (catboost_model, X_train_cat, y_train),
    "LightGBM": (lightgbm_model, X_train, y_train)
}

results = []

for name, predictions in model_predictions.items():
    model, train_features, train_target = fitted_models[name]

    if name == "CatBoost":
        train_predictions = model.predict(train_features).ravel()
    else:
        train_predictions = model.predict(train_features)

    results.append({
        "Model": name,
        "Train Accuracy": accuracy_score(
            train_target, train_predictions
        ),
        "Validation Accuracy": accuracy_score(
            y_val, predictions
        ),
        "Macro F1": f1_score(
            y_val, predictions, average="macro"
        ),
        "Macro Precision": precision_score(
            y_val, predictions, average="macro",
            zero_division=0
        ),
        "Macro Recall": recall_score(
            y_val, predictions, average="macro",
            zero_division=0
        )
    })

comparison = pd.DataFrame(results)

# Sort by validation accuracy
comparison = comparison.sort_values(
    by="Validation Accuracy",
    ascending=False
).reset_index(drop=True)

# Display scores rounded to 4 decimal places
display(
    comparison.style.format({
        "Train Accuracy": "{:.4f}",
        "Validation Accuracy": "{:.4f}",
        "Macro F1": "{:.4f}",
        "Macro Precision": "{:.4f}",
        "Macro Recall": "{:.4f}"
    })
)

print("Best model by validation accuracy:")
print(comparison.loc[0, "Model"])
print(
    "Validation accuracy:",
    round(comparison.loc[0, "Validation Accuracy"], 4)
)

# Save the comparison table
comparison.to_csv("model_comparison.csv", index=False)
print("\nComparison saved to model_comparison.csv")


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,Model,Train Accuracy,Validation Accuracy,Macro F1,Macro Precision,Macro Recall
0,XGBoost Baseline,0.9529,0.8934,0.8832,0.8912,0.8762
1,XGBoost Feature Engineered,0.9529,0.8934,0.8832,0.8912,0.8762
2,CatBoost,0.8959,0.8912,0.8810,0.8895,0.8737
3,LightGBM,0.9998,0.8878,0.8772,0.8858,0.8700


Best model by validation accuracy:
XGBoost Baseline
Validation accuracy: 0.8934

Comparison saved to model_comparison.csv


Tune XGBoost with randomized search

In [ ]:

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from xgboost import XGBClassifier
import numpy as np

xgb_tuning_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=1
    ))
])

param_dist = {
    "model__n_estimators": [200, 300, 400, 500, 600],
    "model__max_depth": [3, 4, 5, 6, 7, 8],
    "model__learning_rate": [0.02, 0.03, 0.05, 0.08, 0.1],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "model__min_child_weight": [1, 3, 5, 7],
    "model__reg_lambda": [1, 2, 5, 10],
    "model__reg_alpha": [0, 0.1, 0.5, 1]
}

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

random_search = RandomizedSearchCV(
    estimator=xgb_tuning_pipeline,
    param_distributions=param_dist,
    n_iter=20,
    scoring="accuracy",
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=1,
    return_train_score=True
)

random_search.fit(X_train, y_train)

print("Best CV accuracy:", round(random_search.best_score_, 4))
print("Best parameters:")
print(random_search.best_params_)


Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best CV accuracy: 0.8827
Best parameters:
{'model__subsample': 0.7, 'model__reg_lambda': 2, 'model__reg_alpha': 1, 'model__n_estimators': 500, 'model__min_child_weight': 5, 'model__max_depth': 5, 'model__learning_rate': 0.03, 'model__colsample_bytree': 1.0}


In [ ]:

from sklearn.metrics import accuracy_score, f1_score
import joblib

# Retrieve the best fitted model from randomized search
xgb_tuned = random_search.best_estimator_

# Evaluate on our original validation set
val_pred_tuned = xgb_tuned.predict(X_val)

tuned_accuracy = accuracy_score(y_val, val_pred_tuned)
tuned_f1 = f1_score(y_val, val_pred_tuned, average="macro")

print("Tuned XGBoost Validation Accuracy:", round(tuned_accuracy, 4))
print("Tuned XGBoost Validation Macro F1:", round(tuned_f1, 4))

# Save the entire fitted pipeline
joblib.dump(
    xgb_tuned,
    "saved_models/xgboost_tuned.joblib"
)

print("\nTuned XGBoost model saved successfully.")


Tuned XGBoost Validation Accuracy: 0.8922
Tuned XGBoost Validation Macro F1: 0.8816

Tuned XGBoost model saved successfully.


In [ ]:

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

# Models and their corresponding input features
models = {
    "XGBoost Baseline": (xgb_model, X_train, X_val),
    "XGBoost Feature Engineered": (
        xgb_fe, X_train_fe, X_val_fe
    ),
    "XGBoost Tuned": (xgb_tuned, X_train, X_val)
}

results = []

for name, (model, train_features, val_features) in models.items():

    # Predictions on training and validation data
    train_predictions = model.predict(train_features)
    val_predictions = model.predict(val_features)

    train_accuracy = accuracy_score(y_train, train_predictions)
    val_accuracy = accuracy_score(y_val, val_predictions)

    results.append({
        "Model": name,
        "Training Accuracy": train_accuracy,
        "Validation Accuracy": val_accuracy,
        "Accuracy Gap": train_accuracy - val_accuracy,
        "Macro F1": f1_score(
            y_val, val_predictions, average="macro"
        ),
        "Macro Precision": precision_score(
            y_val, val_predictions,
            average="macro", zero_division=0
        ),
        "Macro Recall": recall_score(
            y_val, val_predictions,
            average="macro", zero_division=0
        )
    })

comparison_xgb = pd.DataFrame(results)

comparison_xgb = comparison_xgb.sort_values(
    by="Validation Accuracy",
    ascending=False
).reset_index(drop=True)

display(
    comparison_xgb.style.format({
        col: "{:.4f}"
        for col in comparison_xgb.columns
        if col != "Model"
    })
)

print(
    "\nBest model by validation accuracy:",
    comparison_xgb.iloc[0]["Model"]
)

# Save comparison results
comparison_xgb.to_csv(
    "xgboost_model_comparison.csv",
    index=False
)

print("\nComparison saved to xgboost_model_comparison.csv")


,Model,Training Accuracy,Validation Accuracy,Accuracy Gap,Macro F1,Macro Precision,Macro Recall
0,XGBoost Baseline,0.9529,0.8934,0.0595,0.8832,0.8912,0.8762
1,XGBoost Feature Engineered,0.9529,0.8934,0.0595,0.8832,0.8912,0.8762
2,XGBoost Tuned,0.9280,0.8922,0.0358,0.8816,0.8889,0.8752



Best model by validation accuracy: XGBoost Baseline

Comparison saved to xgboost_model_comparison.csv


In [ ]:

import os
import zipfile
import pandas as pd
import joblib

os.makedirs("submissions", exist_ok=True)

# Make sure the trained models are saved
joblib.dump(xgb_model, "saved_models/xgboost_baseline.joblib")
joblib.dump(
    xgb_fe,
    "saved_models/xgboost_feature_engineered.joblib"
)
joblib.dump(xgb_tuned, "saved_models/xgboost_tuned.joblib")

# Model name, model object, and test features
submission_models = {
    "xgboost_baseline": (xgb_model, X_test),
    "xgboost_feature_engineered": (xgb_fe, X_test_fe),
    "xgboost_tuned": (xgb_tuned, X_test)
}

submission_files = []

for name, (model, features) in submission_models.items():

    # Predict the three machine conditions
    predictions = model.predict(features)

    # Create Kaggle submission in the required format
    submission = pd.DataFrame({
        "unit_id": test["unit_id"].values,
        "condition": predictions.astype(int)
    })

    # Save CSV
    filepath = f"submissions/{name}.csv"
    submission.to_csv(filepath, index=False)
    submission_files.append(filepath)

    print(f"\n{name}.csv")
    print("Shape:", submission.shape)
    print("Columns:", submission.columns.tolist())
    print("Missing predictions:", submission["condition"].isna().sum())
    print("Predicted class counts:")
    print(submission["condition"].value_counts().sort_index())

    display(submission.head())

# Package the three CSV files into one ZIP
zip_path = "xgboost_kaggle_submissions.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for filepath in submission_files:
        zf.write(filepath, arcname=os.path.basename(filepath))

print("\nAll three submission files have been generated.")
print("ZIP file:", zip_path)



xgboost_baseline.csv
Shape: (4000, 2)
Columns: ['unit_id', 'condition']
Missing predictions: 0
Predicted class counts:
condition
0    2119
1     944
2     937
Name: count, dtype: int64


,unit_id,condition
0,UNT-184630,0
1,UNT-228473,0
2,UNT-255783,1
3,UNT-119059,2
4,UNT-205559,0



xgboost_feature_engineered.csv
Shape: (4000, 2)
Columns: ['unit_id', 'condition']
Missing predictions: 0
Predicted class counts:
condition
0    2119
1     944
2     937
Name: count, dtype: int64


,unit_id,condition
0,UNT-184630,0
1,UNT-228473,0
2,UNT-255783,1
3,UNT-119059,2
4,UNT-205559,0



xgboost_tuned.csv
Shape: (4000, 2)
Columns: ['unit_id', 'condition']
Missing predictions: 0
Predicted class counts:
condition
0    2094
1     946
2     960
Name: count, dtype: int64


,unit_id,condition
0,UNT-184630,0
1,UNT-228473,0
2,UNT-255783,1
3,UNT-119059,2
4,UNT-205559,0



All three submission files have been generated.
ZIP file: xgboost_kaggle_submissions.zip


In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Prepare validation data for CatBoost
X_val_cat = X_val.copy()
X_val_cat["unit_type"] = (
    X_val_cat["unit_type"].fillna("Unknown").astype(str)
)

# Get class probabilities from both models
xgb_probs = xgb_model.predict_proba(X_val)
cat_probs = catboost_model.predict_proba(X_val_cat)

# Soft voting: average the predicted probabilities
ensemble_probs = (xgb_probs + cat_probs) / 2

# Select the class with the highest probability
ensemble_preds = np.argmax(ensemble_probs, axis=1)

# Compare individual models with the ensemble
results = pd.DataFrame({
    "Model": ["Baseline XGBoost", "CatBoost", "XGBoost + CatBoost Ensemble"],
    "Validation Accuracy": [
        accuracy_score(y_val, xgb_model.predict(X_val)),
        accuracy_score(y_val, catboost_model.predict(X_val_cat)),
        accuracy_score(y_val, ensemble_preds)
    ],
    "Macro F1": [
        f1_score(y_val, xgb_model.predict(X_val), average="macro"),
        f1_score(y_val, catboost_model.predict(X_val_cat), average="macro"),
        f1_score(y_val, ensemble_preds, average="macro")
    ]
})

display(results.round(4))

print("\nEnsemble Classification Report:")
print(classification_report(y_val, ensemble_preds))


,Model,Validation Accuracy,Macro F1
0,Baseline XGBoost,0.8934,0.8832
1,CatBoost,0.8912,0.8810
2,XGBoost + CatBoost Ensemble,0.8931,0.8831



Ensemble Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.94      0.92      1600
           1       0.88      0.84      0.86       800
           2       0.90      0.84      0.87       800

    accuracy                           0.89      3200
   macro avg       0.89      0.88      0.88      3200
weighted avg       0.89      0.89      0.89      3200



In [ ]:

import os
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score

# Prepare test data for CatBoost
X_test_cat = X_test.copy()
X_test_cat["unit_type"] = (
    X_test_cat["unit_type"].fillna("Unknown").astype(str)
)

# Get class probabilities from both models
xgb_test_probs = xgb_model.predict_proba(X_test)
cat_test_probs = catboost_model.predict_proba(X_test_cat)

# Soft voting: average the probabilities
ensemble_test_probs = (xgb_test_probs + cat_test_probs) / 2

# Final predicted classes
ensemble_test_preds = np.argmax(ensemble_test_probs, axis=1)

# Create Kaggle submission
submission_ensemble = pd.DataFrame({
    "unit_id": test["unit_id"].values,
    "condition": ensemble_test_preds.astype(int)
})

# Save CSV
os.makedirs("submissions", exist_ok=True)
submission_path = "submissions/xgboost_catboost_ensemble.csv"
submission_ensemble.to_csv(submission_path, index=False)

# Verify submission
print("Submission shape:", submission_ensemble.shape)
print("Columns:", submission_ensemble.columns.tolist())
print("\nPredicted class distribution:")
print(submission_ensemble["condition"].value_counts().sort_index())
print("\nFirst 5 rows:")
display(submission_ensemble.head())

print("\nSaved to:", submission_path)


Submission shape: (4000, 2)
Columns: ['unit_id', 'condition']

Predicted class distribution:
condition
0    2127
1     934
2     939
Name: count, dtype: int64

First 5 rows:


,unit_id,condition
0,UNT-184630,0
1,UNT-228473,0
2,UNT-255783,1
3,UNT-119059,2
4,UNT-205559,0



Saved to: submissions/xgboost_catboost_ensemble.csv


In [ ]:

import os
import pandas as pd

# Prepare test data for CatBoost
X_test_cat = X_test.copy()
X_test_cat["unit_type"] = (
    X_test_cat["unit_type"].fillna("Unknown").astype(str)
)

# Predict machine conditions
catboost_test_preds = catboost_model.predict(X_test_cat)
catboost_test_preds = catboost_test_preds.reshape(-1).astype(int)

# Create submission dataframe
submission_catboost = pd.DataFrame({
    "unit_id": test["unit_id"].values,
    "condition": catboost_test_preds
})

# Save as CSV
os.makedirs("submissions", exist_ok=True)
submission_path = "submissions/catboost_submission.csv"
submission_catboost.to_csv(submission_path, index=False)

# Verify the submission
print("Submission shape:", submission_catboost.shape)
print("Columns:", submission_catboost.columns.tolist())
print("\nPredicted class distribution:")
print(submission_catboost["condition"].value_counts().sort_index())
display(submission_catboost.head())

print("\nSaved to:", submission_path)


Submission shape: (4000, 2)
Columns: ['unit_id', 'condition']

Predicted class distribution:
condition
0    2132
1     922
2     946
Name: count, dtype: int64


,unit_id,condition
0,UNT-184630,0
1,UNT-228473,0
2,UNT-255783,1
3,UNT-119059,2
4,UNT-205559,0



Saved to: submissions/catboost_submission.csv
